<a href="https://colab.research.google.com/github/chhorponnsovan/ECAM-ROA-I4GIM-S2/blob/main/MultipleRegressionExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile regression_utils.py
import pandas as pd
import statsmodels.api as sm

def display_regression_analysis(df, y_col, X_cols):
    """
    Performs OLS regression and displays the regression summary and a cusle.

    Args:        df (pd.DataFrame): The input DataFrame containing the data.
        y_col (str): The name of the dependent variable column.
        X_cols (list): A list of names of the independent variable columns.
    """
    # Prepare data for statsmodels
    y = df[y_col]
    X = df[X_cols]
    X_sm = sm.add_constant(X)

    # Create and fit the OLS model
    model_sm = sm.OLS(y, X_sm)
    results = model_sm.fit()

    # Calculate ANOVA components
    df_model = results.df_model
    df_residual = results.df_resid
    df_total = df_model + df_residual

    ss_model = results.ess # Explained Sum of Squares (Regression SS)
    ss_residual = results.ssr # Residual Sum of Squares (Error SS)
    ss_total = ss_model + ss_residual # Total Sum of Squares

    ms_model = ss_model / df_model
    ms_residual = ss_residual / df_residual

    f_statistic = results.fvalue
    p_value_f = results.f_pvalue

    # Create a DataFrame for the ANOVA table
    anova_data = {
        'df': [df_model, df_residual, df_total],
        'SS': [ss_model, ss_residual, ss_total],
        'MS': [ms_model, ms_residual, ''],
        'F': [f_statistic, '', ''],
        'Significance F': [p_value_f, '', '']
    }
    anova_table = pd.DataFrame(anova_data, index=['Regression', 'Residual', 'Total'])

    print("\n--- OLS Regression Summary ---")
    print(results.summary())
    print("\n--- Custom ANOVA Table ---")
    # .to_string() helps with formatting in console output
    print(anova_table.round(4).to_string())


Writing regression_utils.py


Now that you've created `regression_utils.py`, you can import the `display_regression_analysis` function from it. This cell demonstrates how you would use it with the `results` object and `anova_table` that are already defined in your current notebook.

**To use this in a new notebook:**
1.  Make sure `regression_utils.py` is in the same directory as your new notebook, or add its directory to your Python path.
2.  Run `from regression_utils import display_regression_analysis`.
3.  Ensure you have your `results` object and `anova_table` (or variables with equivalent content) set up in the new notebook.
4.  Call the function: `display_regression_analysis(results, anova_table)`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Identification
home_id = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

# Dependent Variable
heating_cost = [250, 360, 165, 43, 92, 200, 355, 290, 230, 120, 73, 205, 400, 320, 72, 272, 94, 190, 235, 139]

# Independent Variables
mean_outside_temp = [35, 29, 36, 60, 65, 30, 10, 7, 21, 55, 54, 48, 20, 39, 60, 20, 58, 40, 27, 30]
attic_insulation = [3, 4, 7, 6, 5, 5, 6, 10, 9, 2, 12, 5, 5, 4, 8, 5, 7, 8, 9, 7]
furnace_age = [6, 10, 3, 9, 6, 5, 7, 10, 11, 5, 4, 1, 15, 7, 6, 8, 3, 11, 8, 5]


In [ ]:
data = {
    "Home": home_id,
    "Heating_Cost": heating_cost,
    "Temp": mean_outside_temp,
    "Insulation": attic_insulation,
    "Furnace_Age": furnace_age
}
df = pd.DataFrame(data)


In [ ]:
# Import the function from your newly created library
import regression_utils
from regression_utils import display_regression_analysis

# Now, call the function using the DataFrame and column names
display_regression_analysis(df, 'Heating_Cost', ['Temp', 'Insulation', 'Furnace_Age'])



--- OLS Regression Summary ---
                            OLS Regression Results                            
Dep. Variable:           Heating_Cost   R-squared:                       0.804
Model:                            OLS   Adj. R-squared:                  0.767
Method:                 Least Squares   F-statistic:                     21.90
Date:                Fri, 08 May 2026   Prob (F-statistic):           6.56e-06
Time:                        03:26:16   Log-Likelihood:                -104.80
No. Observations:                  20   AIC:                             217.6
Df Residuals:                      16   BIC:                             221.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         427.

In [ ]:
X = df[['Temp', 'Insulation', 'Furnace_Age']]
y = df['Heating_Cost']

model = LinearRegression()
model.fit(X, y)

intercept = model.intercept_
coefficients = model.coef_

print(f"Regression Equation: y = {intercept:.2f} + {coefficients[0]:.2f}*Temp + {coefficients[1]:.2f}*Insulation + {coefficients[2]:.2f}*Furnace_Age")

Regression Equation: y = 427.19 + -4.58*Temp + -14.83*Insulation + 6.10*Furnace_Age


The **standard error of a coefficient** measures the precision of the coefficient estimate. In simpler terms:

*   **Smaller Standard Error:** Indicates that the estimated coefficient is more precise and less likely to vary if you were to repeat the sampling and regression many times. This means we have more confidence in the estimate's accuracy.
*   **Larger Standard Error:** Suggests that the estimated coefficient is less precise, meaning it could vary significantly if the process were repeated. This implies less confidence in the single estimated value.

It's used in conjunction with the coefficient estimate to construct confidence intervals and calculate t-statistics (which are then used to derive the p-values we discussed earlier). A larger standard error typically leads to a larger p-value, indicating less statistical significance for that particular predictor.

$$\int_{1}^{2} x \,dx$$
$$\begin{pmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{pmatrix}$$
